# Stage 1: reproduce Kandpal et al. on Enron (GPT-Neo 125M)
**Gate (plan section 4):** T̂ AUROC near Kandpal's 0.873, and the null model ≈ chance. Otherwise debug; do not proceed.

**Getting the results:** at the end of a Kaggle run, download `/kaggle/working/outputs/S1_<time>.tar.gz` and extract
it into `Stage1/results/`, which is git-ignored. Every `scores*.csv.gz` and `history_*.json` under it is loaded
(several runs and seeds can sit side by side). This notebook needs no GPU.

In [ ]:
import os, sys, numpy as np, pandas as pd, matplotlib.pyplot as plt
sys.path.insert(0, os.path.abspath(".."))
import analysis as A
RESULTS = os.environ.get("S1_RESULTS", "results")   # folder holding the extracted Kaggle archive(s)
R_DRAWS = 200                  # random draws of m docs per user, for m < 32
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
df, hist = A.load(RESULTS)
summary = df.groupby(["run", "seed"]).agg(docs=("doc_id", "size"), users=("user", "nunique"),
                                         best_epoch=("best_epoch", "first"), model=("model", "first"))
summary["trained_users"] = df[df.trained == 1].groupby(["run", "seed"]).user.nunique()
print(summary)

## 1. Training: validation loss by epoch
`val_in` = members' validation docs (selects the best epoch). `val_out` = non-members' validation docs. The gap between the two is the user-level generalisation gap, E[Δ] (plan Step B.2).

In [ ]:
H = A.history_table(hist)
display(H.round(4))
fig, axes = plt.subplots(1, max(1, H.run.nunique()), figsize=(5 * max(1, H.run.nunique()), 3.2), squeeze=False)
for ax, (tag, h) in zip(axes[0], H.groupby("run")):
    ax.plot(h.epoch, h.val_in, "o-", color=C1, label="val in (members)")
    ax.plot(h.epoch, h.val_out, "s-", color=C2, label="val out (non-members)")
    b = h[h.best]; ax.axvline(b.epoch.iloc[0], color="0.6", ls=":", lw=1)
    ax.set(title=tag, xlabel="epoch (0 = pretrained)", ylabel="NLL per token"); ax.legend(frameon=False, fontsize=8)
plt.tight_layout(); plt.show()

## 2. Score sanity
The mean s(x) should be clearly higher for trained users than for untrained ones (main run). In the null run, EVAL users are all untrained, so their fake labels b should show no gap.

In [ ]:
print(df.groupby(["run", "seed", "trained"]).agg(docs=("s", "size"), mean_s=("s", "mean"), sd_s=("s", "std"),
      mean_tokens=("n_tokens", "mean")).round(3))
print("\nnull run, EVAL users, by fake label b:")
print(df[(df.run == "null") & (df.half == "EVAL")].groupby(["seed", "b"]).s.agg(["size", "mean"]).round(3))

## 3. Main run: T̂ AUROC vs m, certificates
Rows are averaged over the R draws of m documents per user (one score per user per draw). `dkw_threshold` = Ĵ needed for a positive certificate with these n1, n0 (Cor. 1). `tpr_1pct_below_1_over_n0 = True` means TPR at 1% FPR rests on fewer than one false positive, so it is not a certified rate.

In [ ]:
main = {s: d for s, d in df[df.run == "main"].groupby("seed")}
rows = {s: A.auroc_vs_m(d, R=R_DRAWS) for s, d in main.items()}
for s, r in rows.items():
    print(f"seed {s}"); display(r.round(3))
fig, ax = plt.subplots(figsize=(5.5, 3.4))
for s, r in rows.items():
    ax.plot(r.m, r.auroc, "o-", color=C1, alpha=.8, label=f"T̂, seed {s}")
    ax.fill_between(r.m, r.auroc_lo, r.auroc_hi, color=C1, alpha=.12, lw=0)
ax.axhline(A.KANDPAL_AUROC, color=C2, ls="--", lw=1.2, label="Kandpal 0.873")
ax.axhline(.5, color="0.6", lw=.8)
ax.set(xscale="log", xticks=A.M_GRID, xticklabels=A.M_GRID, xlabel="m (held-out docs per user)", ylabel="AUROC", ylim=(.45, 1))
ax.legend(frameon=False, fontsize=8); plt.tight_layout(); plt.show()

In [ ]:
for s, d in main.items():
    r = A.report_m32(d)
    print(f"seed {s}, m = 32:", {k: (np.round(v, 3).tolist() if isinstance(v, (list, float)) else v) for k, v in r.items()})

## 4. Null model (EVAL users only; their labels are fake)
Every statistic must show J ≈ 0 here (plan Step B.4).

In [ ]:
null = {s: d[d.half == "EVAL"] for s, d in df[df.run == "null"].groupby("seed")}
if not null:
    print("no null run yet")
for s, d in null.items():
    print(f"null seed {s}"); display(A.auroc_vs_m(d, R=R_DRAWS).round(3))

## 5. Gate verdict (automatic part)
At m = 32. "Near 0.873" means within ±0.05; that tolerance is our choice. The null passes if its 95% user-bootstrap CI contains 0.5 and its DKW certificate is ≤ 0.
**Caveat:** Kandpal et al.'s 0.873 may use a different number of documents per user than our m = 32. Compare the whole curve, not one point.

In [ ]:
for s in main:
    g = A.gate(main[s], null.get(s))
    print(f"seed {s}:", {k: (np.round(v, 3).tolist() if isinstance(v, (list, float)) else v) for k, v in g.items()})

## Notes (to fill after reading the results)
- Training (best epoch, val gap):
- T̂ AUROC vs m, vs Kandpal:
- Certificates:
- Null model:
- Gate decision: